# Lesson 04 Lab — Why BF16 Is Often the First Low-Precision Choice

**Puzzle:** FP16 and BF16 both use 16 bits. Why can their numerical behavior differ dramatically?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

FP16 and BF16 consume the same two bytes, but they spend those bits differently. BF16 inherits FP32's eight-bit exponent and sacrifices fraction precision; FP16 keeps a longer fraction but only five exponent bits. That trade changes where overflow occurs and how much rounding error accumulates, so a format decision cannot be made from byte count alone.


## 0. Predict before running

1. Predict which 16-bit format represents `1e5` without Inf and which produces the lower GEMM error.
2. Predict whether equal storage implies equal GEMM latency on this GPU.
3. Decide which metric would make you choose FP16 despite BF16's wider range.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

FP16 and BF16 both occupy 16 bits, but FP16 uses 5 exponent and 10 fraction bits whereas BF16 uses 8 exponent and 7 fraction bits. The former offers finer local spacing; the latter offers a much larger dynamic range.

- BF16 keeps an eight-bit exponent, so its range resembles FP32 while its fraction is shorter.
- FP16 has more fraction bits but a much smaller exponent range.
- A stable dtype is not automatically the fastest dtype; measure the actual workload.


## 2. Derive the mechanism

Rounding error is governed by representable spacing near a value, while overflow is governed by exponent range. Accumulation policy adds a third variable: low-precision inputs may still accumulate into a wider type depending on the operator.

A normalized binary floating-point value has the form `(-1)^s × 2^e × (1.f)`. Exponent bits determine dynamic range; fraction bits determine spacing between adjacent representable numbers at a fixed exponent. BF16's range is close to FP32, but its seven stored fraction bits make unit-roundoff much larger than FP16's ten. In a dot product, inputs are rounded before multiplication and partial sums may use a wider accumulator, so input format and accumulation format must be named separately.

This predicts a three-way trade: BF16 should survive large magnitudes, FP16 should often reconstruct ordinary-range values more accurately, and either 16-bit format may use a faster matrix path than FP32. The benchmark tests each axis independently instead of collapsing them into one winner.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "04-bf16-first"
device = require_cuda()
torch.manual_seed(2026 + 4)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | FP32 GEMM and FP32 reference output |
| Candidate | FP16 and BF16 GEMMs on the same 1536×1536 matrices |
| Held constant | shape, random source values, GPU, warm-up, repetitions, comparison reference |
| Measurements | finite-range probe, RMSE/cosine error, median and p90 latency |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare range, matrix-multiplication error, and CUDA timing for FP32, FP16, and BF16.


## 5. Read the experiment code

The lab separates large-value representability, GEMM error, and GEMM latency into three observations so one does not stand in for the others.

The range probe casts `1e5` into both 16-bit formats and records finiteness. The GEMM probe uses the same logical matrices, evaluates output error against FP32, and times each path with twelve post-warm-up CUDA-event samples. Keeping the error and timing records side by side prevents a fast but numerically invalid path from looking successful.

Because the tensors are random and the shape is one square GEMM, the result is a format demonstration rather than a universal training recommendation. Real networks can amplify rounding through normalization, softmax, optimizer state, and long reductions.

Only after these variables match the protocol should the cell be executed.


In [2]:
n = 1536; a32 = torch.randn(n, n, device=device); b32 = torch.randn(n, n, device=device)
ref = a32 @ b32; rows = {}
for dtype in (torch.float32, torch.float16, torch.bfloat16):
    a, b = a32.to(dtype), b32.to(dtype); out = a @ b
    rows[str(dtype).split(".")[-1]] = {"timing": cuda_benchmark(lambda: a @ b, warmup=4, repeats=12),
                                       "error": error_metrics(ref, out.float())}
range_probe = {"fp16_1e5_finite": bool(torch.isfinite(torch.tensor([1e5], device=device).half()).item()),
               "bf16_1e5_finite": bool(torch.isfinite(torch.tensor([1e5], device=device).bfloat16()).item()),
               "fp16_max": torch.finfo(torch.float16).max, "bf16_max": torch.finfo(torch.bfloat16).max}
result = base_result(4, "pytorch-gpu"); result.update({"matrix_shape": [n, n], "formats": rows,
    "range_probe": range_probe, "conclusion": "BF16 preserved the large-value range while FP16 and BF16 showed different accuracy/performance trade-offs."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| FP16 represents 1e5 | no |
| BF16 represents 1e5 | yes |
| FP16 RMSE | 0.014106 |
| BF16 RMSE | 0.112772 |
| FP16 median | 0.044608 ms |
| BF16 median | 0.044160 ms |
| FP32 median | 0.133376 ms |


## 7. Interpret rather than merely print

BF16 represented `1e5` while FP16 overflowed; the recorded maximum finite values were approximately `3.39e38` and `65504`. On the ordinary-range GEMM, FP16 had lower RMSE (0.014106) than BF16 (0.112772), exactly the fraction-bit trade predicted by the format layouts. Median latency was nearly tied—0.044608 ms for FP16 and 0.044160 ms for BF16—while FP32 took 0.133376 ms.

The evidence supports BF16 as a stability-first default for wide-range workloads, not as an accuracy or speed winner in every column. FP16 remained more precise for this input distribution and equally fast within the measured spread.

**Inspection rule:** Look separately at overflow behavior, error against FP32, and latency. No single column decides every workload.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "BF16 preserved the large-value range while FP16 and BF16 showed different accuracy/performance trade-offs.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:14+00:00",
  "formats": {
    "bfloat16": {
      "error": {
        "cosine": 0.99999589,
        "mae": 0.08880086,
        "max_abs": 0.71353149,
        "rmse": 0.11277169
      },
      "timing": {
        "median_ms": 0.04416,
        "p90_ms": 0.044704,
        "repeats": 12,
        "samples_ms": [
          0.045888,
          0.044128,
          0.043488,
          0.0432,
          0.043456,
          0.044192,
          0.044064,
          0.044928,
          0.04432,
          0.044416,
          0.044704,
          0.043872
        ],
        "warmup": 4
      }
    },
    "fl

## 9. Make the bounded decision

> BF16 is a pragmatic stability-first baseline on supported hardware, but workload-specific error and speed still need measurement.

**Acceptance/rollback:** Test both a range probe and workload output error against FP32, then measure latency on the target shape. Keep BF16 only when stability and performance meet the frozen thresholds.

**Failure analysis:** Selecting BF16 solely because it did not overflow can hide unacceptable rounding error; selecting FP16 solely for lower RMSE can fail as soon as activations exceed its range. Another failure is to assume the accumulator shares the input dtype. Record autocast policy and operator behavior when reduction accuracy matters.


## 10. Extend the evidence

Repeat the experiment after scaling inputs across several orders of magnitude and add long reductions, softmax, and layer normalization. For training, compare loss curves and gradient-finiteness rates rather than one GEMM. A useful decision chart marks the magnitude range where FP16 first fails and the error tolerance where BF16 becomes unacceptable.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
